# Amazon Food Review Classification using LSTM

Classify food reviews into score categories (1-5 stars) using an LSTM model trained on only 10% of the dataset.

Key fixes vs. the previous version:
- **10% stratified sample** (keeps minority classes represented) instead of 25%.
- **No text augmentation** - the old `nlpaug` augmentation duplicated minority texts into the training set (causing overfitting and data leakage with the test set). Class weights now handle the imbalance instead.
- **Smaller, regularized model** (single BiLSTM, dropout, L2 weight decay) to fight overfitting.
- **Explicit train/val/test split** so validation is stable and no augmentation leakage inflates the metrics.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, SpatialDropout1D
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

import warnings
warnings.filterwarnings('ignore')

print('TensorFlow version:', tf.__version__)

## 1. Load and Sample 10% of the Data

In [ ]:
DATA_PATH = '/kaggle/input/datasets/shashwathk15/amazon-review/amazon_review.csv'

# Fallback so the notebook also runs locally when the Kaggle path is absent.
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'amazon_review.csv'

df = pd.read_csv(DATA_PATH)
print(f'Full dataset shape: {df.shape}')
print(f'\nScore distribution (full):')
print(df['Score'].value_counts().sort_index())
df.head()

In [ ]:
# Clean: drop rows with missing text or score
df = df.dropna(subset=['Text', 'Score'])
df['Score'] = df['Score'].astype(int)
df['Text'] = df['Text'].astype(str).str.strip()

# Use only 10% of the data, stratified by Score so the minority
# classes (1-3 stars) stay represented in the sample.
df_sampled = pd.concat(
    [cls.sample(frac=0.10, random_state=42) for _, cls in df.groupby('Score', sort=False)],
    ignore_index=True
)

print(f'Sampled dataset shape: {df_sampled.shape}')
print(f'\nScore distribution (10% stratified sample):')
print(df_sampled['Score'].value_counts().sort_index())

In [ ]:
plt.figure(figsize=(8, 4))
df_sampled['Score'].value_counts().sort_index().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Score Distribution (10% Stratified Sample)')
plt.xlabel('Score')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 2. Text Preprocessing

In [ ]:
MAX_VOCAB_SIZE = 10000
MAX_SEQUENCE_LENGTH = 150
EMBEDDING_DIM = 64

texts = df_sampled['Text'].values
labels = (df_sampled['Score'] - 1).values  # shift to 0-indexed
num_classes = 5

print(f'Number of classes: {num_classes}')
print(f'Class names: {np.arange(num_classes) + 1}')

tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
word_index = tokenizer.word_index
print(f'Unique tokens: {len(word_index)}')

X = pad_sequences(sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
y = to_categorical(labels, num_classes=num_classes)

print(f'Input shape: {X.shape}')
print(f'Output shape: {y.shape}')

In [ ]:
# Stratified splits: 70% train, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'Training set:   {X_train.shape[0]} samples')
print(f'Validation set: {X_val.shape[0]} samples')
print(f'Test set:       {X_test.shape[0]} samples')

# Balanced class weights computed ONLY on the training set.
# This is the main tool against the 4/5-star majority dominating training.
y_train_labels = np.argmax(y_train, axis=1)
class_weights_array = compute_class_weight('balanced', classes=np.arange(num_classes), y=y_train_labels)
class_weights = dict(enumerate(class_weights_array))

print(f'\nClass weights (to handle imbalance):')
for i, w in enumerate(class_weights):
    print(f'  Score {i+1}: {class_weights[i]:.3f}')

## 3. Build LSTM Model

In [ ]:
# Smaller, regularized model to reduce overfitting on the 10% sample.
model = Sequential([
    Embedding(MAX_VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_SEQUENCE_LENGTH),
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    Dense(64, activation='relu', kernel_regularizer=l2(1e-4)),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 4. Train the Model

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5, verbose=1)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=128,
    callbacks=[early_stop, reduce_lr],
    class_weight=class_weights,
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Train Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Evaluation - Precision and Recall per Category

In [ ]:
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

score_names = ['1 star', '2 stars', '3 stars', '4 stars', '5 stars']

print('=' * 60)
print('CLASSIFICATION REPORT')
print('=' * 60)
print(classification_report(y_true_classes, y_pred_classes, target_names=score_names, digits=4))

precision, recall, f1, support = precision_recall_fscore_support(
    y_true_classes, y_pred_classes, average=None
)

metrics_df = pd.DataFrame({
    'Score': score_names,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Support': support
})

print('\n' + '=' * 60)
print('PRECISION & RECALL PER CATEGORY')
print('=' * 60)
print(metrics_df.to_string(index=False))

print(f'\nMacro Average Precision: {np.mean(precision):.4f}')
print(f'Macro Average Recall: {np.mean(recall):.4f}')
print(f'Weighted Average Precision: {np.average(precision, weights=support):.4f}')
print(f'Weighted Average Recall: {np.average(recall, weights=support):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_pos = np.arange(len(score_names))
width = 0.35

axes[0].bar(x_pos - width/2, precision, width, label='Precision', color='steelblue', edgecolor='black')
axes[0].bar(x_pos + width/2, recall, width, label='Recall', color='coral', edgecolor='black')
axes[0].set_xlabel('Score Category')
axes[0].set_ylabel('Score')
axes[0].set_title('Precision & Recall per Category')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(score_names)
axes[0].legend()
axes[0].set_ylim(0, 1)

cm = confusion_matrix(y_true_classes, y_pred_classes)
im = axes[1].imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
axes[1].set_title('Confusion Matrix')
plt.colorbar(im, ax=axes[1])
tick_marks = np.arange(num_classes)
axes[1].set_xticks(tick_marks)
axes[1].set_xticklabels(score_names, rotation=45)
axes[1].set_yticks(tick_marks)
axes[1].set_yticklabels(score_names)
axes[1].set_ylabel('True')
axes[1].set_xlabel('Predicted')

fmt = 'd'
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        axes[1].text(j, i, format(cm[i, j], fmt),
                     ha='center', va='center',
                     color='white' if cm[i, j] > thresh else 'black')

plt.tight_layout()
plt.show()

## 6. Overall Accuracy

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy:.4f}')